# HallusionBench — Google Colab 1-Click Runner (A → B → C trên 3 Mô hình)

**Zero-shot evaluation (0 epoch — KHÔNG train/fine-tune).**
- **Cell 1:** Mount Google Drive + kiểm tra GPU / VRAM / BF16 native.
- **Cell 2:** Clone/pull nhánh `benchmark/hallusionbench` + cài thư viện.
- **Cell 3:** Chạy toàn bộ unit tests & E2E tests (`test_hallusionbench.py`).
- **Cell 4 (Phần A):** Tải dataset thật `rayguan/HallusionBench` 1 lần rồi lưu `.tar` vĩnh viễn trên Drive (lần sau giải nén ~2 giây), chạy `audit` & xuất `pilot_ids_8sub.json` phủ đủ **8/8 subcategory**.
- **Cell 5 (Phần B):** Chạy 3 mô hình (`smolvlm2`, `qwen3_vl`, `phi4_multimodal`) theo `STAGE` chọn sẵn (`"pilot"` đủ 8/8 dạng bài | `"full_baseline"` đủ **1.129 câu** | `"smoke_test"` 2 câu). Tự lưu từng mẫu vào Drive & tự resume khi ngắt.
- **Cell 6–7 (Phần C):** Chấm điểm `aAcc / fAcc / qAcc / VD / VS / Easy / Hard`, kiểm chứng `check_results.py` & `check_manifest.py`, xuất bảng ca lỗi `error_cases.md` và bất đồng 3 mô hình.
- **Cell 8:** Nén toàn bộ kết quả thành 1 file `.zip` lưu trên Drive & tải về máy.

## Cell 1 — Kết nối Google Drive & kiểm tra GPU / CUDA / BF16 native

In [ ]:
import os
import sys
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/hallusionbench_runs")
OUTPUT_DIR = DRIVE_ROOT / "results"
AUDIT_DIR = DRIVE_ROOT / "audit_and_pilot"
DRIVE_DATASET_CACHE = DRIVE_ROOT / "dataset_cache"
LOCAL_DATA_ROOT = Path("/content/data/HallusionBench")
LOCAL_HF_CACHE = Path("/content/hf_cache")

for p in (OUTPUT_DIR, AUDIT_DIR, DRIVE_DATASET_CACHE, LOCAL_DATA_ROOT.parent, LOCAL_HF_CACHE):
    p.mkdir(parents=True, exist_ok=True)

os.environ["DATA_ROOT"] = str(LOCAL_DATA_ROOT)
os.environ["HF_HOME"] = str(LOCAL_HF_CACHE)
os.environ["MODEL_CACHE_DIR"] = str(LOCAL_HF_CACHE)
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["AUDIT_DIR"] = str(AUDIT_DIR)
os.environ["DRIVE_DATASET_CACHE"] = str(DRIVE_DATASET_CACHE)

print("Python:", sys.version)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
bf16_emulated = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
try:
    bf16_native = torch.cuda.is_bf16_supported(including_emulation=False) if torch.cuda.is_available() else False
except TypeError:
    bf16_native = bf16_emulated and (torch.cuda.get_device_properties(0).major >= 8)
print(f"GPU count: {torch.cuda.device_count()} | BF16 native: {bf16_native} | BF16 emulated: {bf16_emulated}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | sm_{props.major}{props.minor} | {round(props.total_memory / 2**30, 2)} GiB")
!nvidia-smi

## Cell 2 — Clone/pull nhánh `benchmark/hallusionbench` & cài thư viện

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hoainhan256/vlm-hallucination-benchmark.git"
BRANCH = "benchmark/hallusionbench"
REPO_DIR = Path("/content/vlm-hallucination-benchmark")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

%cd /content/vlm-hallucination-benchmark
!git rev-parse HEAD
!ls -la benchmarks/hallusionbench

missing_core = [p for p in ("vlm_benchmark", "benchmarks/hallusionbench/dataset.py", "benchmarks/hallusionbench/inference.py", "benchmarks/hallusionbench/evaluation.py") if not Path(p).exists()]
if missing_core:
    raise RuntimeError(f"Nhánh {BRANCH} thiếu file cốt lõi: {missing_core}")

!pip install -q -r requirements.txt
!pip install -q --upgrade transformers accelerate qwen-vl-utils num2words peft backoff soundfile

## Cell 3 — Kiểm tra cấu trúc repo & chạy toàn bộ unit tests (A + B + C + E2E)

In [ ]:
import sys
from pathlib import Path

repo_root = str(Path("/content/vlm-hallucination-benchmark").resolve())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from vlm_benchmark.config import validate_all

print("Trạng thái khóa configs chung (official gate):")
for item in validate_all("configs/models.yaml", "configs/datasets.yaml", "configs/experiment.yaml"):
    print(" -", item)

!python scripts/check_repository.py
!python -m pytest -v tests/ benchmarks/hallusionbench/tests/

## Cell 4 (PHẦN A) — Nạp dataset từ Drive (hoặc tải 1 lần lưu lên Drive), Audit & xuất Pilot phủ đủ 8/8 dạng bài

In [ ]:
import json
import os
import shutil
import tarfile
from pathlib import Path
from huggingface_hub import snapshot_download

HF_DATASET_REPO = "rayguan/HallusionBench"
HF_DATASET_REVISION = "2067dfdce5abc7efdbb0927f4721ae282f0b6c51"
LOCAL_DATA_ROOT = Path(os.environ["DATA_ROOT"])
AUDIT_DIR = Path(os.environ["AUDIT_DIR"])
DRIVE_DATASET_TAR = Path(os.environ["DRIVE_DATASET_CACHE"]) / f"HallusionBench_{HF_DATASET_REVISION[:8]}.tar"
AUDIT_REPORT = AUDIT_DIR / "hallusionbench_audit_report.json"
PILOT_FILE = AUDIT_DIR / "pilot_ids_8sub.json"

if not (LOCAL_DATA_ROOT / "HallusionBench.json").is_file():
    if DRIVE_DATASET_TAR.is_file():
        print(f"Nạp dataset lưu sẵn từ Google Drive: {DRIVE_DATASET_TAR}...")
        if LOCAL_DATA_ROOT.exists():
            shutil.rmtree(LOCAL_DATA_ROOT)
        LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        with tarfile.open(DRIVE_DATASET_TAR, "r") as tf:
            tf.extractall(LOCAL_DATA_ROOT)
        print("Giải nén xong!")
    else:
        print(f"Lần đầu: tải {HF_DATASET_REPO} @ {HF_DATASET_REVISION}...")
        snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset", revision=HF_DATASET_REVISION, local_dir=str(LOCAL_DATA_ROOT))
        tmp_tar = Path("/content/HallusionBench_tmp.tar")
        with tarfile.open(tmp_tar, "w") as tf:
            for item in LOCAL_DATA_ROOT.iterdir():
                if item.name != ".cache":
                    tf.add(item, arcname=item.name)
        shutil.move(str(tmp_tar), str(DRIVE_DATASET_TAR))
        print("Đã lưu vĩnh viễn trên Drive:", DRIVE_DATASET_TAR)

os.environ["DATASET_REV"] = HF_DATASET_REVISION
os.environ["PILOT_FILE"] = str(PILOT_FILE)

!python -m benchmarks.hallusionbench.dataset \
  --mode audit \
  --dataset-json "$DATA_ROOT/HallusionBench.json" \
  --data-root "$DATA_ROOT" \
  --dataset-source "rayguan/HallusionBench" \
  --dataset-revision "$DATASET_REV" \
  --output "$AUDIT_DIR/hallusionbench_audit_report.json"

# Xuất tập pilot trọn nhóm phủ ĐỦ cả 8/8 subcategory (chart, figure, illusion, map, math, ocr, table, video)
!python -m benchmarks.hallusionbench.dataset \
  --mode export-pilot \
  --dataset-json "$DATA_ROOT/HallusionBench.json" \
  --data-root "$DATA_ROOT" \
  --dataset-source "rayguan/HallusionBench" \
  --dataset-revision "$DATASET_REV" \
  --target-groups 8 \
  --output "$AUDIT_DIR/pilot_ids_8sub.json"

!python -m benchmarks.hallusionbench.dataset \
  --mode verify-pilot \
  --dataset-json "$DATA_ROOT/HallusionBench.json" \
  --data-root "$DATA_ROOT" \
  --pilot-file "$AUDIT_DIR/pilot_ids_8sub.json"

pilot = json.loads(PILOT_FILE.read_text(encoding="utf-8"))
print(f"\nPilot 8 dạng bài: {pilot['sample_count']} mẫu ({pilot['group_count']} nhóm) | Subcategories: {pilot['breakdown_by_subcategory']}")

## Cell 5 (PHẦN B) — Chạy Inference thật (0 epoch) cho 3 mô hình

Chọn `STAGE`:
- `"pilot"` (mặc định): Chạy tập **8/8 subcategory** (`pilot_ids_8sub.json`), mất ~5 phút/model.
- `"full_baseline"`: Chạy **TOÀN BỘ 1.129 câu / 346 ảnh** (tự lưu từng mẫu vào Drive; nếu Colab ngắt cứ mở lại bấm chạy đúng cell này sẽ tự resume).
- `"smoke_test"`: Chạy nhanh 2 câu kiểm tra kỹ thuật.

In [ ]:
import os
import subprocess
import torch

# === CHỌN PHẠM VI & MÔ HÌNH CHẠY ===
MODELS_TO_RUN = ["smolvlm2", "qwen3_vl", "phi4_multimodal"]
STAGE = "pilot"          # "pilot" (8/8 dạng bài) | "full_baseline" (1.129 câu) | "smoke_test" (2 câu)
MAX_NEW_TOKENS = 128
MAX_SAMPLES = 2          # Chỉ dùng khi STAGE == "smoke_test"

try:
    bf16_native = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    bf16_native = torch.cuda.is_available() and torch.cuda.is_bf16_supported() and (torch.cuda.get_device_properties(0).major >= 8)

DTYPE = "bfloat16" if bf16_native else "float16"
gpu_tag = "bf16gpu" if bf16_native else "t4"
EXPERIMENT_ID = f"hb_colab_{gpu_tag}_{STAGE}"
os.environ["EXPERIMENT_ID"] = EXPERIMENT_ID
os.environ["STAGE"] = STAGE

RUN_DIRS = {}
for model_key in MODELS_TO_RUN:
    attn_impl = "eager" if (model_key == "phi4_multimodal" and not bf16_native) else "sdpa"
    run_id = f"{STAGE}_{model_key}_{DTYPE}"
    run_dir = f"{os.environ['OUTPUT_DIR']}/{EXPERIMENT_ID}/{run_id}"
    RUN_DIRS[model_key] = run_dir

    cmd = [
        "python", "-m", "benchmarks.hallusionbench.inference",
        "--mode", "run",
        "--allow-model-load",
        "--model-key", model_key,
        "--stage", STAGE,
        "--dtype", DTYPE,
        "--attn-implementation", attn_impl,
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
        "--data-root", os.environ["DATA_ROOT"],
        "--dataset-source", "rayguan/HallusionBench",
        "--dataset-revision", os.environ["DATASET_REV"],
        "--experiment-id", EXPERIMENT_ID,
        "--run-id", run_id,
        "--output-root", os.environ["OUTPUT_DIR"],
    ]
    if model_key == "phi4_multimodal":
        cmd.append("--trust-remote-code")
    if STAGE == "smoke_test":
        cmd.extend(["--max-samples", str(MAX_SAMPLES)])
    elif STAGE == "pilot":
        cmd.extend(["--pilot-file", os.environ["PILOT_FILE"]])
    # Nếu STAGE == "full_baseline": không truyền --pilot-file -> chạy đủ 1.129 câu!

    print("\n" + "=" * 70)
    print(f"MODEL: {model_key} | STAGE: {STAGE} | DTYPE: {DTYPE} | ATTN: {attn_impl}")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"--> Xong {model_key} (exit={rc}) | Output: {run_dir}")

## Cell 6 (PHẦN C) — Chấm điểm `aAcc / fAcc / qAcc`, kiểm chứng 7 artifact & in bảng 3 mô hình

In [ ]:
import json
import os
import subprocess
from pathlib import Path

summary_rows = []
for model_key, run_dir_str in RUN_DIRS.items():
    run_dir = Path(run_dir_str)
    status_data = json.loads((run_dir / "run_status.json").read_text(encoding="utf-8"))
    meta_data = json.loads((run_dir / "run_metadata.json").read_text(encoding="utf-8"))

    if status_data.get("status") != "BLOCKED":
        subprocess.run(
            [
                "python", "-m", "benchmarks.hallusionbench.evaluation",
                "--mode", "score",
                "--run-dir", str(run_dir),
                "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
                "--data-root", os.environ["DATA_ROOT"],
            ],
            check=True,
        )

    subprocess.run(["python", "scripts/check_results.py", str(run_dir / "predictions.jsonl")], check=True)
    subprocess.run(["python", "scripts/check_manifest.py", str(run_dir)], check=True)

    metrics_data = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    m = metrics_data.get("metrics") or {}
    c = metrics_data.get("summary_counts") or {}
    summary_rows.append({
        "model": model_key,
        "status": status_data.get("status"),
        "dtype": meta_data.get("dtype"),
        "model_rev": str(meta_data.get("resolved_model_revision"))[:10],
        "scored_q": c.get("scored_questions", 0),
        "aAcc": m.get("aAcc"),
        "fAcc": m.get("fAcc"),
        "qAcc": m.get("qAcc"),
        "errors": c.get("error_records", 0),
    })

print(f"\n=== BẢNG TỔNG HỢP 3 MÔ HÌNH (STAGE: {os.environ['STAGE']}) ===")
print("| Model | Status | Dtype | Scored Q | aAcc | fAcc | qAcc | Errors | Model SHA |")
print("|---|---|---|---:|---:|---:|---:|---:|---|")
for r in summary_rows:
    print(f"| `{r['model']}` | `{r['status']}` | `{r['dtype']}` | {r['scored_q']} | {r['aAcc']} | {r['fAcc']} | {r['qAcc']} | {r['errors']} | `{r['model_rev']}` |")

## Cell 7 (PHẦN C) — Xuất bảng ca lỗi 6 nhãn & tìm ca bất đồng giữa 3 mô hình

In [ ]:
import json
import os
import subprocess
from pathlib import Path

completed_pairs = []
for model_key, run_dir_str in RUN_DIRS.items():
    run_dir = Path(run_dir_str)
    if json.loads((run_dir / "run_status.json").read_text(encoding="utf-8")).get("status") == "BLOCKED":
        continue
    completed_pairs.append(f"{model_key}={run_dir}")
    subprocess.run(
        [
            "python", "-m", "benchmarks.hallusionbench.evaluation",
            "--mode", "extract",
            "--run-dir", str(run_dir),
            "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
            "--data-root", os.environ["DATA_ROOT"],
        ],
        check=True,
    )

if len(completed_pairs) >= 2:
    disagree_path = Path(os.environ["OUTPUT_DIR"]) / os.environ["EXPERIMENT_ID"] / "cross_model_disagreements.json"
    subprocess.run(
        [
            "python", "-m", "benchmarks.hallusionbench.evaluation",
            "--mode", "disagreements",
            "--run-dirs", *completed_pairs,
            "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
            "--data-root", os.environ["DATA_ROOT"],
            "--output", str(disagree_path),
        ],
        check=True,
    )
    print("Ca bất đồng giữa các mô hình:", len(json.loads(disagree_path.read_text(encoding="utf-8"))))

## Cell 8 — Gom toàn bộ kết quả thành 1 file `.zip` & tải về máy tính

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

exp_id = os.environ["EXPERIMENT_ID"]
exp_dir = Path(os.environ["OUTPUT_DIR"]) / exp_id
audit_dir = Path(os.environ["AUDIT_DIR"])
bundle_zip = Path(os.environ["OUTPUT_DIR"]) / f"hallusionbench_results_{exp_id}.zip"

with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for src_dir, prefix in ((audit_dir, "audit_and_pilot"), (exp_dir, exp_id)):
        if not src_dir.exists():
            continue
        for path in sorted(src_dir.rglob("*")):
            if path.is_file():
                zf.write(path, arcname=f"{prefix}/{path.relative_to(src_dir)}")

print(f"Đã đóng gói ({round(bundle_zip.stat().st_size / 1024, 1)} KB): {bundle_zip}")
files.download(str(bundle_zip))